# NoisiQ — Week 8 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: May 11 – May 17, 2026*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 8 of the NoisiQ project.

By the end of this notebook you should be able to:
- Import any Qiskit `QuantumCircuit` directly into NoisiQ without rebuilding the circuit
- Load a circuit from a QASM file or QASM string
- Run NoisiQ noise simulation on an imported Qiskit circuit
- Visualize the error propagation from a circuit originally built in Qiskit

---

## Week 8 Milestone Summary

Week 8 implements the **Qiskit adapter** — the bridge that lets users bring any existing
Qiskit circuit into NoisiQ. The goal is full gate coverage: any gate Qiskit supports
should be importable. Unsupported gates raise a clear `NotImplementedError`.
QASM 2.0 file and string import is also included via Qiskit's parser.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `noisiq/adapters/__init__.py` — adapter submodule public API | TJ | ☐ Todo |
| `noisiq/adapters/gate_map.py` — Qiskit gate name → NoisiQ Gate mapping | TJ | ☐ Todo |
| `noisiq/adapters/qiskit_adapter.py` — `from_qiskit(qc)` implementation | TJ | ☐ Todo |
| `noisiq/adapters/qasm_adapter.py` — `from_qasm_file()` and `from_qasm_string()` | TJ | ☐ Todo |
| `tests/adapters/test_qiskit_adapter.py` — round-trip tests | TJ | ☐ Todo |
| `tests/adapters/test_qasm_adapter.py` — QASM parse tests | TJ | ☐ Todo |
| All tests passing via `pytest` | TJ | ☐ Todo |
| CI passing on GitHub | TJ | ☐ Todo |
| Week 8 demo section of this notebook complete | TJ | ☐ Todo |

---

## File Build Order

```
1. noisiq/adapters/gate_map.py          ← Gate mapping table (depends on ir/gates.py)
2. noisiq/adapters/qiskit_adapter.py    ← Main adapter (depends on gate_map + ir/circuit)
3. noisiq/adapters/qasm_adapter.py      ← QASM parser (depends on qiskit_adapter)
4. noisiq/adapters/__init__.py          ← Exposes from_qiskit, from_qasm_file, from_qasm_string
5. tests/adapters/test_qiskit_adapter.py
6. tests/adapters/test_qasm_adapter.py
```

---

## Technical Requirements for Week 8

**Gate coverage in `gate_map.py`** (minimum required set):
- Single-qubit: `h`, `x`, `y`, `z`, `s`, `sdg`, `t`, `tdg`, `id`
- Parameterized single-qubit: `rx(θ)`, `ry(θ)`, `rz(θ)`, `u1(λ)`, `u2(φ,λ)`, `u3(θ,φ,λ)`
- Two-qubit: `cx` / `cnot`, `cz`, `cy`, `swap`, `iswap`
- Three-qubit: `ccx` (Toffoli), `cswap` (Fredkin)
- Unsupported gates: raise `NotImplementedError(f"Gate '{name}' not yet supported in NoisiQ")`

**Timestep assignment:**
- Use Qiskit's `QuantumCircuit.data` order as the timestep sequence
- Multi-qubit gates increment timestep for all involved qubits

**QASM parsing:**
- Parse via `qiskit.qasm2.load(path)` or `qiskit.qasm2.loads(string)`
- Then call `from_qiskit()` on the resulting `QuantumCircuit`

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| 2026-05-11 | Week 8 started. Full Qiskit gate coverage is the priority. | DS |
| | | |

# Installation Instructions (Developer)

```bash
pip install -e .
pip install qiskit   # Qiskit must be installed for the adapter to work
```

In [ ]:
import noisiq as nq
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT

print(f"noisiq {nq.__version__} imported successfully")

# 1. Import a Bell state circuit built in Qiskit
print("\n--- Bell state: Qiskit → NoisiQ ---")
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

noisiq_bell = nq.adapters.from_qiskit(qc_bell)
print(f"Imported circuit: {noisiq_bell}")
print(f"Gates: {[op.gate.name for op in noisiq_bell.operations]}")

# 2. Import a 4-qubit QFT circuit
print("\n--- 4-qubit QFT: Qiskit → NoisiQ ---")
qc_qft = QFT(4).decompose()
noisiq_qft = nq.adapters.from_qiskit(qc_qft)
print(f"QFT circuit imported: {len(noisiq_qft.operations)} gates on {noisiq_qft.n_qubits} qubits")

# 3. Run noise simulation on the imported QFT circuit
print("\n--- Noise simulation on imported QFT ---")
preset = nq.noise.SuperconductingPreset()
noise = preset.to_noise_model()

backend = nq.backends.BackendSelector.select(noisiq_qft, noise)
result = backend.run(noisiq_qft, noise_model=noise, n_shots=500)
nq.visualization.plot_error_heatmap(result, noisiq_qft)

In [ ]:
# 4. Import from QASM string
print("--- QASM string import ---")
qasm_str = """
OPENQASM 2.0;
include \"qelib1.inc\";
qreg q[3];
h q[0];
cx q[0], q[1];
cx q[1], q[2];
"""

ghz_circuit = nq.adapters.from_qasm_string(qasm_str)
print(f"GHZ circuit from QASM: {ghz_circuit}")
nq.visualization.draw_circuit(ghz_circuit)